# Parcial de BackTracking — 2026-2

## Instrucciones generales

- Queda **terminantemente prohibido** el uso de cualquier asistente de inteligencia artificial (ChatGPT, GitHub Copilot, Claude, Gemini, etc.), así como copiar o adaptar soluciones encontradas en internet. Todo el código entregado debe ser producido individualmente por el estudiante durante el parcial.
- El profesor podrá **citar a cualquier estudiante a sustentar** su solución de forma oral. No poder explicar y defender el código entregado — o no presentarse a la sustentación — anulará el punto correspondiente.
- El uso de **typehints** en todas las funciones es obligatorio.
- Todo ejercicio debe incluir el **análisis de complejidad temporal** de la solución entregada (use la sección *"El pastel de la complejidad"* como guía para construirlo).
- Cualquier intento de fraude anulará el examen.
- El parcial debe enviarse como notebook (**.ipynb**, para que no sea bloqueado por el filtro de correo) **adjunto** — puede enviarse desde cualquier dirección de correo — a **johnathancalle@gmail.com**.

---
## El pastel de la complejidad: cómo analizar un backtracking

Una forma útil de imaginar cualquier algoritmo de backtracking es como un **pastel de capas**: cada **capa** del pastel corresponde a un **nivel de decisión** de la recursión (por ejemplo, "elegir el siguiente movimiento", "elegir la siguiente letra", "elegir la siguiente zancada"), y cada capa está dividida en **porciones**, una por cada opción que el algoritmo explora en ese nivel.

Para calcular la complejidad de un backtracking, siga esta receta en cinco pasos:

1. **Factor de ramificación (b):** ¿cuántas opciones distintas se exploran en cada llamada recursiva? Puede ser constante (p. ej. b = 3 si siempre hay tres movimientos posibles) o variar según el estado (p. ej. depende de cuántas letras queden disponibles en un banco).
2. **Profundidad del árbol (d):** ¿cuántas decisiones como máximo se toman antes de llegar a un caso base? Suele estar acotada por el tamaño de la entrada (número de casillas, longitud del nombre, número de escalones).
3. **Tamaño del árbol (número de nodos):** sin poda, el número de llamadas recursivas está acotado por la suma de las porciones de todas las capas, es decir, por O(b^d) cuando la ramificación es constante. Esta es la "masa" del pastel completo.
4. **Trabajo por nodo (w):** ¿cuánto cuesta cada llamada individual sin contar las llamadas recursivas que genera (copiar un camino, validar una restricción, construir una cadena)? Ese costo se multiplica por el número de nodos.
5. **Poda (pruning):** si el algoritmo descarta ramas completas antes de explorarlas (por ejemplo, al superar el límite K de paradas, o al agotar las copias disponibles de una letra), la poda **no cambia la complejidad del peor caso** a menos que se demuestre formalmente que reduce el árbol de forma asintótica — pero sí reduce el trabajo real observado. En su análisis indique explícitamente qué poda(s) implementó y si afecta o no la cota superior teórica.

La complejidad temporal total queda entonces como **T(n) = O(nodos_del_árbol × trabajo_por_nodo)**. Este análisis (b, d, poda y w) es el que se espera para **cada uno de los tres ejercicios** de este parcial.

---
## Ejercicio 1 — Rutas de DroniCafé

Una empresa de logística del Eje Cafetero, **DroniCafé**, usa drones autónomos para transportar café recién tostado entre bodegas dispuestas en una cuadrícula de n×m. Cada dron inicia su recorrido en la bodega de la esquina superior izquierda (fila 0, columna 0) y debe llegar a la bodega de despacho final, en la esquina inferior derecha (fila n-1, columna m-1), moviéndose **únicamente hacia la derecha o hacia abajo**, una casilla a la vez.

Algunas casillas de la cuadrícula son **puntos de control fitosanitario obligatorios**, marcados con la letra `'P'` en la matriz; las demás casillas se marcan con `'.'`. Detenerse en un punto de control consume tiempo y batería, así que, por directriz de la empresa, **ninguna ruta puede pasar por más de K puntos de control**.

Implemente, usando **backtracking**, una función que reciba la cuadrícula y el valor de K, y retorne **todas** las rutas válidas desde (0,0) hasta (n-1, m-1) que cumplan la restricción de máximo K paradas.

### Casos de prueba — Ejercicio 1

Estos datos no se evalúan automáticamente: úselos para probar su propia implementación (usted define el nombre de la función, los parámetros y el formato de retorno) y compare manualmente contra el resultado esperado.

In [ ]:
# Caso de prueba 1: cuadrícula 3x3 con dos puntos de control en la columna 1, k=1
cuadricula_1 = [
    ['.', 'P', '.'],
    ['.', 'P', '.'],
    ['.', '.', '.'],
]
k_1 = 1
# Resultado esperado: 4 rutas válidas
#   (0,0) -> (0,1) -> (0,2) -> (1,2) -> (2,2)
#   (0,0) -> (1,0) -> (1,1) -> (1,2) -> (2,2)
#   (0,0) -> (1,0) -> (1,1) -> (2,1) -> (2,2)
#   (0,0) -> (1,0) -> (2,0) -> (2,1) -> (2,2)

# Caso de prueba 2: cuadrícula 2x4 con dos puntos de control consecutivos en la fila 1, k=1
cuadricula_2 = [
    ['.', '.', '.', '.'],
    ['.', 'P', 'P', '.'],
]
k_2 = 1
# Resultado esperado: 2 rutas válidas
#   (0,0) -> (0,1) -> (0,2) -> (1,2) -> (1,3)
#   (0,0) -> (0,1) -> (0,2) -> (0,3) -> (1,3)

---
## Ejercicio 2 — Los nombres más costosos de Verbum Lab

**Verbum Lab** es un estudio de *branding* que cobra a sus clientes según el "costo creativo" de cada letra usada en el nombre de marca que diseña: letras que exigen procesos de fundición tipográfica especiales tienen un costo positivo, mientras que letras muy comunes —que el estudio reutiliza de proyectos anteriores— tienen costo negativo y funcionan como descuento. El costo de cada letra está dado por un diccionario `costos: dict[str, int]`.

Para un nuevo proyecto, el cliente exige que el nombre tenga **exactamente `longitud` letras**, usando únicamente las letras disponibles en el banco tipográfico del estudio: un inventario `banco: dict[str, int]` que indica cuántas copias de cada letra hay disponibles (una vez usada una copia dentro de un nombre, no puede reutilizarse en ese mismo nombre).

Implemente, usando **backtracking**, una función que genere todos los nombres posibles de la longitud pedida que respeten las cantidades disponibles en el banco, y determine cuál (o cuáles, si hay empate) tienen el **mayor costo total**.

### Casos de prueba — Ejercicio 2

Estos datos no se evalúan automáticamente: úselos para probar su propia implementación (usted define el nombre de la función, los parámetros y el formato de retorno) y compare manualmente contra el resultado esperado.

In [ ]:
# Caso de prueba 1
costos_1 = {'a': 5, 'b': -3, 'c': 8, 'd': 2}
banco_1 = {'a': 1, 'b': 1, 'c': 2, 'd': 1}
longitud_1 = 3
# Resultado esperado: costo máximo = 21
# Nombres que lo alcanzan (dos copias de 'c' y una de 'a'): "cca", "cac", "acc"

# Caso de prueba 2
costos_2 = {'a': -1, 'b': 4, 'c': -5, 'd': 6, 'e': 3}
banco_2 = {'a': 2, 'b': 1, 'c': 1, 'd': 1, 'e': 2}
longitud_2 = 2
# Resultado esperado: costo máximo = 10
# Nombres que lo alcanzan (una 'd' y una 'b'): "db", "bd"

---
## Ejercicio 3 — Subiendo a la Piedra del Peñol

La **Piedra del Peñol**, en el oriente antioqueño, se sube por una escalinata empotrada en una grieta natural de la roca, que se dice tiene varios cientos de escalones repartidos en tramos. En cada tramo, quien sube puede avanzar **1, 2 o 3 escalones** de una sola zancada.

Los guías turísticos recomiendan **no dar tres o más zancadas consecutivas del mismo tamaño**, para no sobrecargar siempre el mismo grupo muscular (sí está permitido repetir un tamaño de zancada, siempre que no sea más de dos veces seguidas).

Implemente, usando **backtracking**, una función que reciba el número total de escalones `n` y retorne **todas** las secuencias de zancadas (listas de valores en {1, 2, 3}) que suman exactamente `n` y que no repiten el mismo tamaño de zancada más de dos veces consecutivas.

### Casos de prueba — Ejercicio 3

Estos datos no se evalúan automáticamente: úselos para probar su propia implementación (usted define el nombre de la función y los parámetros) y compare manualmente contra el resultado esperado.

In [ ]:
# Caso de prueba 1
n_1 = 4
# Resultado esperado: 6 secuencias válidas (se excluye [1,1,1,1] por tener 4 repeticiones consecutivas)
#   [1,1,2]  [1,2,1]  [2,1,1]  [1,3]  [3,1]  [2,2]

# Caso de prueba 2
n_2 = 5
# Resultado esperado: 10 secuencias válidas
#   [1,1,2,1] [1,2,1,1] [1,1,3] [1,3,1] [1,2,2] [2,1,2] [2,2,1] [2,3] [3,1,1] [3,2]